In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import pandas as pd

split_path = '/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/raw_3'
train_pl = pd.read_csv(f'{split_path}/train.csv')
test_1_pl = pd.read_csv(f'{split_path}/test_p1.csv')
test_2_pl = pd.read_csv(f'{split_path}/test_p2.csv')
test_3_pl = pd.read_csv(f'{split_path}/test_p3.csv')
test_4_pl = pd.read_csv(f'{split_path}/test_p4.csv')
val = pd.read_csv(f'{split_path}/val_p4.csv')

In [19]:
import pandas as pd
import polars as pl

print('Checking for non-numeric columns in train_pl:')
string_columns = []
for col in train_pl.columns:
    if not pd.api.types.is_numeric_dtype(train_pl[col].dtype):
        print(f"Column '{col}' has type: {train_pl[col].dtype}")
        string_columns.append(col)

if not string_columns:
    print("No non-numeric columns found that need type conversion.")
else:
    print(f"Found non-numeric columns: {string_columns}. Please review if these need to be converted to a numeric type for median imputation.")

Checking for non-numeric columns in train_pl:
No non-numeric columns found that need type conversion.


In [22]:
median_values = {}
for col in train_pl.columns:
    if pd.api.types.is_numeric_dtype(train_pl[col]):
        median_values[col] = train_pl[col].median()

print("Calculated median values for numeric columns in train_pl.")

Calculated median values for numeric columns in train_pl.


In [24]:
import pandas as pd # Ensure pandas is imported if not already
# import polars as pl # Polars is not needed for this corrected cell

dataframes_for_imputation = [
    (train_pl, 'train_pl'),
    (test_1_pl, 'test_1_pl'),
    (test_2_pl, 'test_2_pl'),
    (test_3_pl, 'test_3_pl'),
    (test_4_pl, 'test_4_pl'),
    (val, 'val')
]

for df_obj, df_name in dataframes_for_imputation:
    print(f"Imputing missing values in {df_name}...")
    # Select only numeric columns that have missing values and their medians were calculated
    columns_to_impute = [
        col for col in df_obj.columns
        if pd.api.types.is_numeric_dtype(df_obj[col].dtype) and df_obj[col].isnull().any() and col in median_values
    ]

    if columns_to_impute:
        # Create a dictionary of median values for the columns to be imputed
        medians_for_current_df = {col: median_values[col] for col in columns_to_impute}
        # Impute missing values using Pandas fillna
        df_obj = df_obj.fillna(medians_for_current_df)
    else:
        print(f"No numeric columns with missing values found in {df_name} for imputation.")

    # Update the global variable with the modified DataFrame
    if df_name == 'train_pl':
        train_pl = df_obj
    elif df_name == 'test_1_pl':
        test_1_pl = df_obj
    elif df_name == 'test_2_pl':
        test_2_pl = df_obj
    elif df_name == 'test_3_pl':
        test_3_pl = df_obj
    elif df_name == 'test_4_pl':
        test_4_pl = df_obj
    elif df_name == 'val':
        val = df_obj
    print(f"Finished imputing missing values in {df_name}.")

print("Median imputation complete for all specified DataFrames.")

Imputing missing values in train_pl...
Finished imputing missing values in train_pl.
Imputing missing values in test_1_pl...
Finished imputing missing values in test_1_pl.
Imputing missing values in test_2_pl...
Finished imputing missing values in test_2_pl.
Imputing missing values in test_3_pl...
Finished imputing missing values in test_3_pl.
Imputing missing values in test_4_pl...
Finished imputing missing values in test_4_pl.
Imputing missing values in val...
Finished imputing missing values in val.
Median imputation complete for all specified DataFrames.


In [26]:
all_datasets = [
    (train_pl, 'train'),
    (val, 'val'),
    (test_1_pl, 'test_1'),
    (test_2_pl, 'test_2'),
    (test_3_pl, 'test_3'),
    (test_4_pl, 'test_4')
]

for df_obj, df_name in all_datasets:
    print(f"Processing {df_name}...")

    # Extract target variable 'label_3' using Pandas syntax
    y = df_obj['label_3']

    # Define columns to be dropped from features
    cols_to_drop = ['user_id', 'course_id', 'label_3']

    # Create feature set X by dropping specified columns using Pandas syntax
    X = df_obj.drop(columns=cols_to_drop)

    # Assign X and y to global variables with appropriate names
    globals()[f'X_{df_name}'] = X
    globals()[f'y_{df_name}'] = y

    print(f"Finished processing {df_name}. X_{df_name} shape: {X.shape}, y_{df_name} shape: {y.shape}")

print("Feature and target separation complete for all datasets.")

Processing train...
Finished processing train. X_train shape: (1859619, 179), y_train shape: (1859619,)
Processing val...
Finished processing val. X_val shape: (232452, 179), y_val shape: (232452,)
Processing test_1...
Finished processing test_1. X_test_1 shape: (232453, 179), y_test_1 shape: (232453,)
Processing test_2...
Finished processing test_2. X_test_2 shape: (232453, 179), y_test_2 shape: (232453,)
Processing test_3...
Finished processing test_3. X_test_3 shape: (232453, 179), y_test_3 shape: (232453,)
Processing test_4...
Finished processing test_4. X_test_4 shape: (232453, 179), y_test_4 shape: (232453,)
Feature and target separation complete for all datasets.


In [27]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

print("Training the model...")
model.fit(X_train, y_train)
print("Model trained successfully.")

Training the model...
Model trained successfully.


In [29]:
print("Making predictions on validation and test sets using NumPy arrays...")

# Make predictions on the validation set
y_pred_val = model.predict(X_val)

# Make predictions on the test sets
y_pred_test_1 = model.predict(X_test_1)
y_pred_test_2 = model.predict(X_test_2)
y_pred_test_3 = model.predict(X_test_3)
y_pred_test_4 = model.predict(X_test_4)

print("Predictions made successfully on all datasets with NumPy arrays.")

Making predictions on validation and test sets using NumPy arrays...
Predictions made successfully on all datasets with NumPy arrays.


In [31]:
#print accuracy
from sklearn.metrics import accuracy_score
import numpy as np

# Round the continuous predictions to the nearest integer for classification evaluation
y_pred_val_rounded = np.round(y_pred_val)
y_pred_test_1_rounded = np.round(y_pred_test_1)
y_pred_test_2_rounded = np.round(y_pred_test_2)
y_pred_test_3_rounded = np.round(y_pred_test_3)
y_pred_test_4_rounded = np.round(y_pred_test_4)

print('Accuracy on validation set:', accuracy_score(y_val, y_pred_val_rounded))
print('Accuracy on test set 1:', accuracy_score(y_test_1, y_pred_test_1_rounded))
print('Accuracy on test set 2:', accuracy_score(y_test_2, y_pred_test_2_rounded))
print('Accuracy on test set 3:', accuracy_score(y_test_3, y_pred_test_3_rounded))
print('Accuracy on test set 4:', accuracy_score(y_test_4, y_pred_test_4_rounded))

Accuracy on validation set: 0.9963003114621514
Accuracy on test set 1: 1.2905834727880474e-05
Accuracy on test set 2: 9.034084309516332e-05
Accuracy on test set 3: 0.005756002288634691
Accuracy on test set 4: 0.99640787600074


In [32]:
X_train

,course_duration,start_year,start_month,start_day,enrollment_to_end,gender,num_course_order,avg_interval_enroll_time,max_interval_enroll_time,min_interval_enroll_time,...,cmt_p4_max_comment_length_phase4,cmt_p4_text_diversity_phase4,cmt_p4_comment_days_active_phase4,cmt_p4_entropy_time_mean_phase4,cmt_p4_most_common_time_bin_phase4,cmt_p4_time_entropy_var_phase4,cmt_p4_positive_ratio_phase4,cmt_p4_neutral_ratio_phase4,cmt_p4_negative_ratio_phase4,cmt_p4_sentiment_entropy_phase4
0,176,2020,2,6,170,2.0,6,2.058000,9.56,0.00,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
1,114,2020,10,30,76,0.0,58,7.192982,59.22,0.02,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
2,170,2020,2,12,69,0.0,56,7.414727,51.91,0.19,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
3,151,2020,2,16,138,2.0,7,7.661667,34.84,0.01,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
4,173,2020,10,30,140,2.0,2,71.060000,71.06,71.06,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859614,91,2019,12,31,52,0.0,49,7.943542,33.74,0.12,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
1859615,121,2020,9,1,31,0.0,60,6.899322,55.12,0.02,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
1859616,93,2020,4,13,91,2.0,2,63.480000,63.48,63.48,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
1859617,121,2020,9,1,37,1.0,3,0.025000,0.05,0.00,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0


In [33]:
output_path = '/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_im_3'

In [38]:
import os
os.makedirs(output_path, exist_ok=True)

In [39]:
train_pl.to_csv(f'{output_path}/train.csv', index=False)
val.to_csv(f'{output_path}/val.csv', index=False)
test_1_pl.to_csv(f'{output_path}/test_1.csv', index=False)
test_2_pl.to_csv(f'{output_path}/test_2.csv', index=False)
test_3_pl.to_csv(f'{output_path}/test_3.csv', index=False)
test_4_pl.to_csv(f'{output_path}/test_4.csv', index=False)